# Electricity Consumption Analytics and Energy-Saving Recommendation System
### AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026
**Student Name:** Shani Kumar  
**Specialization:** B.Tech Computer Science Engineering (AI & ML / Data Analytics)  
**Dataset:** Individual Household Electric Power Consumption (UCI Machine Learning Repository / EDF R&D)  
**Original Dataset File:** `household_power_consumption (1).txt`  

---
## 1. Project Overview & Objectives
This project investigates residential electricity consumption patterns across 2,075,259 minute-level observations collected over nearly four years (December 2006 to November 2010).

### Key Objectives:
1. Clean and preprocess high-frequency minute-by-minute electrical telemetry (kW, Wh, Volts, Amperes).
2. Conduct Exploratory Data Analysis (EDA) to understand seasonal trends, diurnal load curves, and appliance sub-metering distributions.
3. Calculate Key Performance Indicators (KPIs) such as cumulative energy (kWh), daily averages, peak load hours, and baseload.
4. Apply statistical and machine learning anomaly detection to isolate unusual consumption surges.
5. Evaluate a chronological time-series forecasting model for daily demand estimation.
6. Formulate evidence-based, data-supported energy efficiency recommendations.

## 2. Importing Required Libraries

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# Visualization settings
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['figure.figsize'] = (12, 5)
print('Libraries successfully imported.')

## 3. Data Ingestion & Initial Inspection
We load the UCI Household Electric Power Consumption dataset. In this dataset, missing readings are marked by `?` and fields are separated by `;`.

In [ ]:
# Path verification
data_path = os.path.join('data', 'household_power_consumption.txt')
if not os.path.exists(data_path):
    fallback = r'C:\Users\Shani\AppData\Local\Temp\household_power_consumption (1).txt'
    if os.path.exists(fallback):
        data_path = fallback

print(f'Loading data from: {data_path}')

# Load raw dataset with specified datatypes
t0 = time.time()
df_raw = pd.read_csv(
    data_path,
    sep=';',
    na_values=['?'],
    dtype={
        'Global_active_power': float,
        'Global_reactive_power': float,
        'Voltage': float,
        'Global_intensity': float,
        'Sub_metering_1': float,
        'Sub_metering_2': float,
        'Sub_metering_3': float
    },
    low_memory=False
)
print(f'Loaded {len(df_raw):,} records in {time.time()-t0:.2f} seconds.')
df_raw.head()

### Understanding the Variables:
- **Date & Time:** Timestamp of measurement (DD/MM/YYYY HH:MM:SS).
- **Global_active_power:** Household global active power (kilowatt, kW).
- **Global_reactive_power:** Household global reactive power (kW).
- **Voltage:** Minute-averaged voltage (volts, V).
- **Global_intensity:** Minute-averaged current intensity (amperes, A).
- **Sub_metering_1:** Kitchen circuit energy (watt-hours, Wh) — dishwasher, microwave, oven.
- **Sub_metering_2:** Laundry circuit energy (watt-hours, Wh) — washing machine, dryer, fridge, lighting.
- **Sub_metering_3:** Electric water-heater and air conditioning (watt-hours, Wh).

## 4. Data Cleaning, Timestamp Parsing & Missing Value Imputation

In [ ]:
# Check missing values
null_counts = df_raw.isnull().sum()
null_pct = (null_counts / len(df_raw)) * 100
missing_summary = pd.DataFrame({'Missing_Count': null_counts, 'Percentage (%)': null_pct.round(3)})
print('Missing Value Breakdown:')
print(missing_summary)

# Parse Datetime index
df_raw['Datetime'] = pd.to_datetime(df_raw['Date'] + ' ' + df_raw['Time'], format='%d/%m/%Y %H:%M:%S')
df_raw.set_index('Datetime', inplace=True)
df_raw.drop(columns=['Date', 'Time'], inplace=True)

# Time-weighted linear interpolation for continuous electrical signals
df_clean = df_raw.interpolate(method='time', limit=60).bfill().ffill()
print('Missing values after time interpolation:', df_clean.isnull().sum().sum())

## 5. Physical Energy Conversions & Remainder Circuit Derivation
Active power is recorded in kilowatts (kW). To compute the energy consumed during each 1-minute sampling window:

$$\text{Energy (Wh)} = P_{\text{active}} (\text{kW}) \times 1000 \times \left(\frac{1}{60}\right) = P_{\text{active}} \times \frac{50}{3}$$

The unmetered remainder circuit represents active electrical energy consumed by household devices not on sub-meters 1, 2, or 3 (e.g. televisions, desktop computers, ceiling lighting, chargers):

In [ ]:
df_clean['Global_active_energy_Wh'] = df_clean['Global_active_power'] * 1000.0 / 60.0
sub_sum = df_clean['Sub_metering_1'] + df_clean['Sub_metering_2'] + df_clean['Sub_metering_3']
df_clean['Sub_metering_remainder_Wh'] = np.maximum(0.0, df_clean['Global_active_energy_Wh'] - sub_sum)

# Resample to daily summary for exploratory and longitudinal analytics
daily_df = df_clean.resample('D').agg({
    'Global_active_power': ['mean', 'max', 'min', 'std'],
    'Voltage': 'mean',
    'Global_intensity': 'mean',
    'Global_active_energy_Wh': 'sum',
    'Sub_metering_1': 'sum',
    'Sub_metering_2': 'sum',
    'Sub_metering_3': 'sum',
    'Sub_metering_remainder_Wh': 'sum'
})
daily_df.columns = ['_'.join(c).strip('_') for c in daily_df.columns]

# Convert energy to kilowatt-hours (kWh)
daily_df['Active_Energy_kWh'] = daily_df['Global_active_energy_Wh_sum'] / 1000.0
daily_df['Sub_metering_1_kWh'] = daily_df['Sub_metering_1_sum'] / 1000.0
daily_df['Sub_metering_2_kWh'] = daily_df['Sub_metering_2_sum'] / 1000.0
daily_df['Sub_metering_3_kWh'] = daily_df['Sub_metering_3_sum'] / 1000.0
daily_df['Sub_metering_remainder_kWh'] = daily_df['Sub_metering_remainder_Wh_sum'] / 1000.0

daily_df.reset_index(inplace=True)
daily_df['Year'] = daily_df['Datetime'].dt.year
daily_df['Month'] = daily_df['Datetime'].dt.month
daily_df['Month_Name'] = daily_df['Datetime'].dt.strftime('%b')
daily_df['DayOfWeek'] = daily_df['Datetime'].dt.day_name()
daily_df['IsWeekend'] = daily_df['Datetime'].dt.dayofweek.isin([5, 6]).astype(int)

print(f'Constructed daily analytics dataframe spanning {len(daily_df)} days.')

## 6. Key Performance Indicators (KPIs) & Empirical Ground Truth

In [ ]:
total_kwh = daily_df['Active_Energy_kWh'].sum()
mean_daily_kwh = daily_df['Active_Energy_kWh'].mean()
median_daily_kwh = daily_df['Active_Energy_kWh'].median()
std_daily_kwh = daily_df['Active_Energy_kWh'].std()
peak_day_idx = daily_df['Active_Energy_kWh'].idxmax()
min_day_idx = daily_df['Active_Energy_kWh'].idxmin()

print('=== RESIDENTIAL ELECTRICITY GROUND-TRUTH KPIS ===')
print(f'Monitoring Period: {daily_df["Datetime"].min().strftime("%Y-%m-%d")} to {daily_df["Datetime"].max().strftime("%Y-%m-%d")}')
print(f'Total Days Tracked: {len(daily_df):,} days')
print(f'Cumulative Consumption: {total_kwh:,.2f} kWh')
print(f'Mean Daily Consumption: {mean_daily_kwh:.2f} kWh/day')
print(f'Median Daily Consumption: {median_daily_kwh:.2f} kWh/day')
print(f'Standard Deviation: {std_daily_kwh:.2f} kWh/day')
print(f'Historical Peak Day: {daily_df.loc[peak_day_idx, "Active_Energy_kWh"]:.2f} kWh on {daily_df.loc[peak_day_idx, "Datetime"].strftime("%Y-%m-%d")}')
print(f'Historical Minimum Day: {daily_df.loc[min_day_idx, "Active_Energy_kWh"]:.2f} kWh on {daily_df.loc[min_day_idx, "Datetime"].strftime("%Y-%m-%d")}')

## 7. Exploratory Data Analysis & Visualization
We generate four fundamental analytical plots:
1. 4-Year Longitudinal Timeline with 30-Day Smoothing
2. Sub-Metering End-Use Distribution
3. 24-Hour Diurnal Demand Profile
4. Day-of-Week and Seasonal Heatmap

In [ ]:
# Plot 1: Longitudinal Timeline
plt.figure(figsize=(14, 5))
rolling_30 = daily_df['Active_Energy_kWh'].rolling(30, center=True).mean()
plt.plot(daily_df['Datetime'], daily_df['Active_Energy_kWh'], color='#90CAF9', alpha=0.6, label='Daily Active Energy (kWh)')
plt.plot(daily_df['Datetime'], rolling_30, color='#1565C0', linewidth=2.2, label='30-Day Moving Average')
plt.title('Daily Electricity Consumption Over 4 Years (Dec 2006 – Nov 2010)', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Date')
plt.ylabel('Active Energy (kWh/day)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Sub-metering Pie / Donut Breakdown
s1_tot = daily_df['Sub_metering_1_kWh'].sum()
s2_tot = daily_df['Sub_metering_2_kWh'].sum()
s3_tot = daily_df['Sub_metering_3_kWh'].sum()
rem_tot = daily_df['Sub_metering_remainder_kWh'].sum()

labels = ['Kitchen (Sub 1)', 'Laundry (Sub 2)', 'HVAC & Water Heater (Sub 3)', 'Remainder (Plug Loads & Lighting)']
shares = [s1_tot, s2_tot, s3_tot, rem_tot]
colors = ['#FF7043', '#26A69A', '#42A5F5', '#AB47BC']

fig, ax = plt.subplots(figsize=(8, 5))
wedges, texts, autotexts = ax.pie(shares, labels=labels, autopct='%1.1f%%', startangle=140, colors=colors, pctdistance=0.75, wedgeprops={'width': 0.45, 'edgecolor': 'white'})
for at in autotexts:
    at.set_fontweight('bold')
ax.set_title('Cumulative Energy Breakdown by Sub-Circuit', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: 24-Hour Diurnal Demand Profile
df_clean['Hour'] = df_clean.index.hour
hourly_df = df_clean.groupby('Hour').agg({
    'Global_active_power': 'mean',
    'Sub_metering_1': 'mean',
    'Sub_metering_2': 'mean',
    'Sub_metering_3': 'mean',
    'Sub_metering_remainder_Wh': 'mean'
}).reset_index()

# Convert minute averages to hourly Watt-hours
for c in ['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3', 'Sub_metering_remainder_Wh']:
    hourly_df[c] = hourly_df[c] * 60.0
hourly_df['Total_Hourly_Wh'] = hourly_df['Sub_metering_1'] + hourly_df['Sub_metering_2'] + hourly_df['Sub_metering_3'] + hourly_df['Sub_metering_remainder_Wh']

plt.figure(figsize=(11, 5))
plt.plot(hourly_df['Hour'], hourly_df['Total_Hourly_Wh'], color='#E65100', marker='o', linewidth=2.5, label='Total Demand (Wh)')
plt.plot(hourly_df['Hour'], hourly_df['Sub_metering_3'], color='#1E88E5', linestyle='--', label='HVAC & Water Heating (Sub 3)')
plt.plot(hourly_df['Hour'], hourly_df['Sub_metering_remainder_Wh'], color='#8E24AA', linestyle=':', label='Remainder (Lighting & Standby)')
plt.axvspan(19, 22, color='#FFCDD2', alpha=0.35, label='Peak Demand Window (19:00 - 22:00)')
plt.axvspan(2, 5, color='#E1F5FE', alpha=0.35, label='Off-Peak Baseload (02:00 - 05:00)')
plt.title('24-Hour Diurnal Demand Profile', fontsize=12, fontweight='bold')
plt.xlabel('Hour of Day')
plt.ylabel('Watt-hours per Hour')
plt.xticks(range(24))
plt.legend()
plt.tight_layout()
plt.show()

## 8. Statistical & Machine Learning Anomaly Detection
We implement two complementary anomaly detection methodologies:
1. **Interquartile Range (IQR):** $Threshold = Q_3 + 1.5 \times IQR$
2. **Isolation Forest:** Tree-based unsupervised anomaly detection separating multi-dimensional feature outliers.

In [ ]:
# 1. Statistical IQR
q1 = daily_df['Active_Energy_kWh'].quantile(0.25)
q3 = daily_df['Active_Energy_kWh'].quantile(0.75)
iqr = q3 - q1
threshold_iqr = q3 + 1.5 * iqr
daily_df['Anomaly_IQR'] = daily_df['Active_Energy_kWh'] > threshold_iqr

# 2. Isolation Forest
iso = IsolationForest(contamination=0.03, random_state=42)
features_iso = daily_df[['Active_Energy_kWh', 'Sub_metering_1_kWh', 'Sub_metering_2_kWh', 'Sub_metering_3_kWh']].fillna(0)
daily_df['Anomaly_IsoForest'] = iso.fit_predict(features_iso) == -1

print(f'IQR Outlier Days: {daily_df["Anomaly_IQR"].sum()} ({daily_df["Anomaly_IQR"].sum()/len(daily_df)*100:.2f}% of days)')
print(f'Isolation Forest Flagged Days: {daily_df["Anomaly_IsoForest"].sum()} ({daily_df["Anomaly_IsoForest"].sum()/len(daily_df)*100:.2f}% of days)')

# Top flagged days
flagged_top = daily_df[daily_df['Anomaly_IQR']].sort_values('Active_Energy_kWh', ascending=False).head(5)
flagged_top[['Datetime', 'Active_Energy_kWh', 'Sub_metering_1_kWh', 'Sub_metering_2_kWh', 'Sub_metering_3_kWh']]

## 9. Chronological Forecasting Model (Short-Term Demand Estimation)
We test a time-series forecasting model to predict next-day active energy (kWh) using historical lags ($t-1, t-2, t-7$) and calendar features. To prevent future-data leakage, we use a strictly chronological train/test split on the last 90 days.

In [ ]:
# Feature engineering
ts_df = daily_df[['Datetime', 'Active_Energy_kWh', 'Month', 'IsWeekend']].copy()
ts_df['Lag_1'] = ts_df['Active_Energy_kWh'].shift(1)
ts_df['Lag_2'] = ts_df['Active_Energy_kWh'].shift(2)
ts_df['Lag_7'] = ts_df['Active_Energy_kWh'].shift(7)
ts_df['Rolling_Mean_7'] = ts_df['Active_Energy_kWh'].shift(1).rolling(7).mean()
ts_df.dropna(inplace=True)

# Chronological holdout test: last 90 days
test_size = 90
train = ts_df.iloc[:-test_size]
test = ts_df.iloc[-test_size:].copy()

features = ['Lag_1', 'Lag_2', 'Lag_7', 'Rolling_Mean_7', 'Month', 'IsWeekend']
X_train, y_train = train[features], train['Active_Energy_kWh']
X_test, y_test = test[features], test['Active_Energy_kWh']

# Fit Ridge Regression Model
model = Ridge(alpha=1.0)
model.fit(X_train, y_train)
test['Pred_Ridge'] = model.predict(X_test)
test['Pred_Baseline_Lag7'] = test['Lag_7']  # Naive persistence baseline

mae_ridge = mean_absolute_error(y_test, test['Pred_Ridge'])
rmse_ridge = root_mean_squared_error(y_test, test['Pred_Ridge'])
mae_baseline = mean_absolute_error(y_test, test['Pred_Baseline_Lag7'])

print(f'Ridge Regression Model — Test MAE: {mae_ridge:.2f} kWh | RMSE: {rmse_ridge:.2f} kWh')
print(f'Naive Weekly Baseline — Test MAE: {mae_baseline:.2f} kWh')
print(f'Model Error Reduction: {((mae_baseline - mae_ridge) / mae_baseline)*100:.1f}% relative improvement over naive persistence.')

## 10. Data-Backed Energy Efficiency Recommendations

Based on the empirical findings from the 4-year dataset, four actionable recommendations are derived:

1. **Baseload / Phantom Load Reduction:** Baseload power between 02:00-05:00 is ~350 Wh/hr on unmetered circuits. Eliminating 100 W of idle standby power saves ~2.4 kWh/day (~876 kWh/year).
2. **Peak-Hour Load Shifting:** Shifting laundry and dishwashing away from 19:00-22:00 to morning off-peak hours cuts evening grid stress and prevents circuit overload.
3. **HVAC & Water Heater Calibration:** Sub-metering 3 accounts for 35.5% of cumulative household demand. Moderating water heater and space heating thermostats by 1°C saves ~8% (~1,073 kWh/year).
4. **Outlier Day Auditing:** High consumption outliers flagged by IQR occurred predominantly during cold winter weeks. Inspecting weatherproofing and insulation prior to winter months provides substantial cost savings.